In [ ]:
# ===== XGBoost TRAINING =====
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score

print("Loading ADASYN data...")
X_train = pd.read_pickle('adasyn_X_train.pkl')
y_train = np.load('adasyn_y_train.npy')
X_val = pd.read_pickle('adasyn_X_val.pkl')
y_val = np.load('adasyn_y_val.npy')

print(f"Training on: {X_train.shape}")

# XGBoost model
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

# Train
print("Training XGBoost...")
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

# Results
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]

print("\n📊 RESULTS")
print(classification_report(y_val, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_val, y_proba):.4f}")

# Save model
model.save_model('xgboost_model.json')
print("\n✅ MODEL SAVED: xgboost_model.json")

# Test predictions
X_test = pd.read_pickle('engineered_test.pkl').select_dtypes([np.number]).fillna(-999)
test_proba = model.predict_proba(X_test)[:, 1]
pd.DataFrame({'TransactionID': X_test.index, 'isFraud': test_proba}).to_csv('predictions.csv', index=False)
print("✅ Predictions saved: predictions.csv")


Loading ADASYN data...
Training on: (682531, 418)
Training XGBoost...
[0]	validation_0-logloss:0.41194
[50]	validation_0-logloss:0.14682
[100]	validation_0-logloss:0.10998
[150]	validation_0-logloss:0.09723
[200]	validation_0-logloss:0.09101
[250]	validation_0-logloss:0.08706
[299]	validation_0-logloss:0.08415

📊 RESULTS
              precision    recall  f1-score   support

           0       0.98      1.00      0.99    113975
           1       0.87      0.42      0.56      4133

    accuracy                           0.98    118108
   macro avg       0.92      0.71      0.78    118108
weighted avg       0.98      0.98      0.97    118108

ROC-AUC: 0.9129

✅ MODEL SAVED: xgboost_model.json
✅ Predictions saved: predictions.csv
